# Notebook 02: End-to-End Training & Evaluation Walkthrough
**Project:** COVID-CT Deep Learning Classifier (`covid-ct-dashboard`)


> [!WARNING]
> **EDUCATIONAL & PORTFOLIO PROJECT ONLY • NOT A MEDICAL DEVICE**  
> *This notebook, associated models, and data pipelines are strictly for research, algorithmic benchmarking, and portfolio demonstration. They are NOT certified by the FDA, CE, or any medical regulatory agency and must NEVER be used for clinical decision making, diagnosis, triage, or patient treatment.*


### Overview:
This notebook demonstrates the complete PyTorch training workflow:
1. Loading leakage-aware split tables (`data/splits/train.csv`, `val.csv`, `test.csv`).
2. Dataset class & augmentation pipeline (RandomResizedCrop, Rotation, Jitter, ImageNet normalization).
3. Instantiating pretrained transfer learning backbones (`MobileNetV3-Large`, `ResNet-18`, `EfficientNet-B0`).
4. Optimizer (AdamW), Cosine Annealing learning rate schedule, and Early Stopping.
5. Training progression inspection across epochs.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import torch

PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SPLITS_DIR, CHECKPOINTS_DIR, get_device, CLASS_NAMES
from src.dataset import CTScanDataset, get_train_transforms, get_eval_transforms
from src.model import build_model, count_parameters

device = get_device()
print('Active Device:', device)

## 1. Inspecting Dataset Splits

In [ ]:
train_df = pd.read_csv(SPLITS_DIR / 'train.csv')
val_df = pd.read_csv(SPLITS_DIR / 'val.csv')
test_df = pd.read_csv(SPLITS_DIR / 'test.csv')

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# Verify strict group separation
train_g = set(train_df['group_id'])
test_g = set(test_df['group_id'])
overlap = train_g.intersection(test_g)
print('Group overlap between Train and Test:', len(overlap))

## 2. Model Architecture Setup

In [ ]:
for name in ['mobilenet_v3_large', 'resnet18', 'efficientnet_b0']:
    m = build_model(name, num_classes=3, pretrained=True)
    total, trainable = count_parameters(m)
    print(f'{name:<20}: {total:>10,} total params | {trainable:>10,} trainable')

## 3. Training History & Benchmark Comparison

In [ ]:
history_csv = PROJECT_ROOT / 'powerbi/data/training_history.csv'
comp_csv = PROJECT_ROOT / 'powerbi/data/model_comparison.csv'

if history_csv.exists():
    h_df = pd.read_csv(history_csv)
    display(h_df.head(10))

if comp_csv.exists():
    c_df = pd.read_csv(comp_csv)
    display(c_df[c_df['split'] == 'test'])